In [1]:
import subprocess

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

# 检查显存
import torch
print(f"\nPyTorch CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {total:.1f} GB")

    if total < 30:
        print("\n WARNING: 显存 < 30GB，将自动切换到 Qwen2.5-Coder-3B + INT8")
    else:
        print("\n 显存充足，可以跑 Qwen2.5-Coder-7B FP16")

Sun May  3 01:36:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   33C    P0             53W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Drive上的模型缓存目录
DRIVE_MODEL_DIR = '/content/drive/MyDrive/vllm_bench/models'
DRIVE_RESULTS_DIR = '/content/drive/MyDrive/vllm_bench/results'

os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)

print(f"Drive模型目录: {DRIVE_MODEL_DIR}")
print(f"Drive结果目录: {DRIVE_RESULTS_DIR}")
print(f"\n当前Drive目录内容:")
!ls -lh {DRIVE_MODEL_DIR}

Mounted at /content/drive
Drive模型目录: /content/drive/MyDrive/vllm_bench/models
Drive结果目录: /content/drive/MyDrive/vllm_bench/results

当前Drive目录内容:
total 4.0K
drwx------ 2 root root 4.0K Mar 17 04:18 Qwen2.5-Coder-7B


In [3]:
import torch

total_vram = torch.cuda.get_device_properties(0).total_memory / 1024**3

# ── 自动选择模型配置 ──────────────────────────────────────
if total_vram >= 38:   # A100 40GB
    MODEL_ID     = "Qwen/Qwen2.5-Coder-7B-Instruct"
    MODEL_NAME   = "Qwen2.5-Coder-7B"
    DTYPE        = "float16"
    QUANTIZATION = None        # 不量化
    MAX_MODEL_LEN = 8192
    GPU_MEM_UTIL  = 0.88

elif total_vram >= 22:  # A100 25GB (Colab)
    MODEL_ID     = "Qwen/Qwen2.5-Coder-7B-Instruct"
    MODEL_NAME   = "Qwen2.5-Coder-7B-INT8"
    DTYPE        = "float16"
    QUANTIZATION = "bitsandbytes"  # 8bit量化，省约一半显存
    MAX_MODEL_LEN = 4096           # 缩短以省KV cache显存
    GPU_MEM_UTIL  = 0.90

else:                   # 其他低显存环境
    MODEL_ID     = "Qwen/Qwen2.5-Coder-3B-Instruct"
    MODEL_NAME   = "Qwen2.5-Coder-3B"
    DTYPE        = "float16"
    QUANTIZATION = None
    MAX_MODEL_LEN = 4096
    GPU_MEM_UTIL  = 0.88
# Drive 上的本地路径
LOCAL_MODEL_PATH = f"{DRIVE_MODEL_DIR}/{MODEL_NAME}"

print("=" * 50)
print(f"GPU VRAM:       {total_vram:.1f} GB")
print(f"选择模型:       {MODEL_ID}")
print(f"量化方式:       {QUANTIZATION or 'None (FP16)'}")
print(f"Max seq len:    {MAX_MODEL_LEN}")
print(f"本地缓存路径:   {LOCAL_MODEL_PATH}")
print("=" * 50)

GPU VRAM:       79.3 GB
选择模型:       Qwen/Qwen2.5-Coder-7B-Instruct
量化方式:       None (FP16)
Max seq len:    8192
本地缓存路径:   /content/drive/MyDrive/vllm_bench/models/Qwen2.5-Coder-7B


In [4]:
# ══ 快速启动 Cell══

from google.colab import drive
drive.mount('/content/drive')

import subprocess, os

def run(cmd):
    proc = subprocess.Popen(
        cmd, shell=True, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True
    )
    for line in proc.stdout:
        print(line, end='')
    proc.wait()

DRIVE_DIR = '/content/drive/MyDrive/vllm_bench'
REQ_FILE  = f'{DRIVE_DIR}/requirements.txt'
os.makedirs(DRIVE_DIR, exist_ok=True)

# ── 判断是否已有 requirements.txt ──────────────────────────
if os.path.exists(REQ_FILE):
    print(f"从 Drive 读取依赖: {REQ_FILE}")
    run(f"pip install numpy==1.26.4 pandas==2.2.2 --force-reinstall --no-cache-dir -q")
    run(f"pip install -r {REQ_FILE} --no-cache-dir -q")
else:
    print("首次安装，按顺序装依赖...")
    run("pip install numpy==1.26.4 pandas==2.2.2 --force-reinstall --no-cache-dir -q")
    run("pip install vllm==0.7.3 nest_asyncio --no-cache-dir -q")

    # 保存 requirements 到 Drive
    with open(REQ_FILE, 'w') as f:
        f.write("vllm==0.7.3\n")
        f.write("transformers==4.48.2\n")
        f.write("tokenizers==0.21.0\n")
        f.write("nest_asyncio\n")
        f.write("aiohttp\n")
    print(f"requirements.txt 已保存到 Drive: {REQ_FILE}")

import nest_asyncio
nest_asyncio.apply()

import numpy, pandas, vllm
print(f"numpy {numpy.__version__} | pandas {pandas.__version__} | vllm {vllm.__version__}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
首次安装，按顺序装依赖...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 395.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 403.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.9/229.9 kB 393.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.5/510.5 kB 399.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.3/349.3 kB 390.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [5]:

import torch
import torchvision

print(torch.__version__)
print(torchvision.__version__)


import vllm
print(f"vLLM version: {vllm.__version__}")

RuntimeError: operator torchvision::nms does not exist

In [ ]:
import os
from huggingface_hub import snapshot_download

def check_model_cached(path):
    """检查模型是否完整缓存（验证关键文件存在）"""
    required = ['config.json', 'tokenizer.json']
    if not os.path.exists(path):
        return False
    files = os.listdir(path)
    has_required = all(f in files for f in required)
    has_weights   = any(f.endswith('.safetensors') for f in files)
    return has_required and has_weights


if check_model_cached(LOCAL_MODEL_PATH):
    print(f"模型已缓存在 Drive: {LOCAL_MODEL_PATH}")
    print(f"   文件列表:")
    for f in sorted(os.listdir(LOCAL_MODEL_PATH)):
        size = os.path.getsize(f"{LOCAL_MODEL_PATH}/{f}") / 1024**3
        print(f"   {f:50s} {size:.2f} GB" if size > 0.01 else f"   {f}")
    USE_MODEL_PATH = LOCAL_MODEL_PATH

else:
    print(f"Drive 中未找到缓存，开始下载 {MODEL_ID}...")
    print(f"   目标路径: {LOCAL_MODEL_PATH}")
    print(f"   预计大小: {'~15GB (7B)' if '7B' in MODEL_ID else '~6GB (3B)'}")
    print(f"   预计时间: 10-20 分钟\n")

    os.makedirs(LOCAL_MODEL_PATH, exist_ok=True)

    # 优先用镜像站
    try:
        snapshot_download(
            repo_id=MODEL_ID,
            local_dir=LOCAL_MODEL_PATH,
            local_dir_use_symlinks=False,
            endpoint="https://hf-mirror.com",   # 镜像站
            ignore_patterns=["*.pt", "*.bin"],   # 只下载safetensors
        )
    except Exception as e:
        print(f"镜像站失败({e})，切换官方站...")
        snapshot_download(
            repo_id=MODEL_ID,
            local_dir=LOCAL_MODEL_PATH,
            local_dir_use_symlinks=False,
            ignore_patterns=["*.pt", "*.bin"],
        )

    print(f"\n下载完成，已保存到 Drive")
    USE_MODEL_PATH = LOCAL_MODEL_PATH

print(f"\n将使用模型路径: {USE_MODEL_PATH}")

In [ ]:
# workload
import random
import json

# ── 三种代码生成场景的 ISL/OSL 分布 ──────────────────────
WORKLOAD_PROFILES = {
    "inline_completion": {
        # Copilot式补全：粘贴整个文件，补全少量代码
        "ISL_range": (512, 2048),    # 长prompt（受限于MAX_MODEL_LEN）
        "OSL_range": (20, 150),      # 短输出
        "weight":    0.50,
        "TTFT_SLO":  1.0,            # 秒，用户感知阈值
        "description": "粘贴文件上下文，补全少量代码"
    },
    "code_explanation": {
        # 解释代码逻辑
        "ISL_range": (256, 1024),
        "OSL_range": (150, 400),
        "weight":    0.25,
        "TTFT_SLO":  2.0,
        "description": "解释代码功能和逻辑"
    },
    "function_generation": {
        # 根据注释生成完整函数
        "ISL_range": (128, 512),
        "OSL_range": (200, 600),     # 最长输出
        "weight":    0.25,
        "TTFT_SLO":  1.5,
        "description": "根据注释生成完整函数或类"
    }
}

# ── 生成 Prompt（用重复token模拟指定长度）─────────────────
def generate_prompt(isl: int) -> str:
    """生成约 isl token 长度的代码 prompt"""
    # 用真实代码片段模板（每个约20token）
    code_snippet = "def process_data(x):\n    # TODO: implement\n    pass\n"
    base = "# Python code context\n" + code_snippet * (isl // 20)
    return base[:isl * 4]  # 粗略按4 char/token截断


def sample_workload(profile_name: str, n: int, seed: int = 42) -> list:
    """按指定profile采样n个请求"""
    random.seed(seed)
    profile = WORKLOAD_PROFILES[profile_name]
    requests = []
    for _ in range(n):
        isl = random.randint(*profile["ISL_range"])
        osl = random.randint(*profile["OSL_range"])
        requests.append({
            "prompt":       generate_prompt(isl),
            "max_tokens":   osl,
            "target_isl":   isl,
            "target_osl":   osl,
            "profile":      profile_name,
        })
    return requests


def sample_mixed_workload(n: int, seed: int = 42) -> list:
    """按权重混合三种场景，模拟真实流量"""
    random.seed(seed)
    profiles = list(WORKLOAD_PROFILES.keys())
    weights  = [WORKLOAD_PROFILES[p]["weight"] for p in profiles]
    requests = []
    for _ in range(n):
        profile = random.choices(profiles, weights=weights)[0]
        p = WORKLOAD_PROFILES[profile]
        isl = random.randint(*p["ISL_range"])
        osl = random.randint(*p["OSL_range"])
        requests.append({
            "prompt":     generate_prompt(isl),
            "max_tokens": osl,
            "target_isl": isl,
            "target_osl": osl,
            "profile":    profile,
        })
    return requests


# 预览各场景分布
print("Workload Profile 预览")
print("=" * 55)
for name, p in WORKLOAD_PROFILES.items():
    print(f"\n{name}")
    print(f"   说明:   {p['description']}")
    print(f"   ISL:    {p['ISL_range'][0]} ~ {p['ISL_range'][1]} tokens")
    print(f"   OSL:    {p['OSL_range'][0]} ~ {p['OSL_range'][1]} tokens")
    print(f"   权重:   {p['weight']*100:.0f}%")
    print(f"   SLO:    TTFT < {p['TTFT_SLO']}s")

# 采样验证
sample = sample_mixed_workload(n=50)
print(f"\n混合workload采样示例 (n=50):")
from collections import Counter
dist = Counter(r['profile'] for r in sample)
for k, v in dist.items():
    print(f"   {k}: {v} 请求 ({v/50*100:.0f}%)")

In [ ]:
import subprocess
import time
import requests as req
import os


SERVER_PORT = 8000
SERVER_LOG  = "/tmp/vllm_server.log"

def build_server_cmd():
    cmd = [
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model",                USE_MODEL_PATH,
        "--served-model-name",    "qwen-coder",
        "--dtype",                DTYPE,
        "--max-model-len",        str(MAX_MODEL_LEN),
        "--gpu-memory-utilization", str(GPU_MEM_UTIL),
        "--port",                 str(SERVER_PORT),
        "--host",                 "0.0.0.0",
        "--disable-log-requests",
    ]
    if QUANTIZATION:
        cmd += ["--quantization", QUANTIZATION]
    return cmd


def wait_for_server(timeout=180):
    """等待服务就绪"""
    url = f"http://localhost:{SERVER_PORT}/health"
    start = time.time()
    while time.time() - start < timeout:
        try:
            r = req.get(url, timeout=2)
            if r.status_code == 200:
                return True
        except:
            pass
        elapsed = time.time() - start
        print(f"\r等待服务启动... {elapsed:.0f}s / {timeout}s", end='')
        time.sleep(5)
    return False


# 启动服务
print("启动 vLLM 服务...")
print(f"模型: {USE_MODEL_PATH}")
print(f"量化: {QUANTIZATION or 'None'}")
print(f"日志: {SERVER_LOG}\n")

log_f = open(SERVER_LOG, 'w')
server_proc = subprocess.Popen(
    build_server_cmd(),
    stdout=log_f,
    stderr=log_f,
)

# 等待就绪
if wait_for_server(timeout=180):
    print(f"\n服务启动成功！监听端口 {SERVER_PORT}")
else:
    print(f"\n服务启动超时，查看日志：")
    !tail -30 {SERVER_LOG}

# 验证推理
test_resp = req.post(
    f"http://localhost:{SERVER_PORT}/v1/chat/completions",
    json={
        "model": "qwen-coder",
        "messages": [{"role": "user", "content": "Write a Python hello world"}],
        "max_tokens": 50,
    }
)
reply = test_resp.json()["choices"][0]["message"]["content"]
print(f"\n推理验证: {reply[:100]}...")
print("推理正常")

In [ ]:
import time
import asyncio
import aiohttp
import numpy as np
from dataclasses import dataclass, field
from typing import List, Optional

@dataclass
class RequestResult:
    profile:       str
    target_isl:    int
    target_osl:    int
    ttft:          Optional[float]  # 秒
    total_time:    Optional[float]  # 秒
    output_tokens: Optional[int]
    tpot:          Optional[float]  # 秒/token
    success:       bool
    error:         str = ""


async def send_request(
    session: aiohttp.ClientSession,
    request: dict,
    server_url: str,
) -> RequestResult:
    """发送单个请求，使用流式输出精确测量TTFT"""
    payload = {
        "model":      "qwen-coder",
        "messages":   [{"role": "user", "content": request["prompt"]}],
        "max_tokens": request["max_tokens"],
        "stream":     True,   # 流式输出，才能准确测TTFT
    }

    ttft = None
    output_tokens = 0
    t_start = time.perf_counter()

    try:
        async with session.post(
            f"{server_url}/v1/chat/completions",
            json=payload,
            timeout=aiohttp.ClientTimeout(total=120)
        ) as resp:
            async for chunk in resp.content:
                line = chunk.decode().strip()
                if line.startswith("data: ") and line != "data: [DONE]":
                    if ttft is None:
                        ttft = time.perf_counter() - t_start  # 第一个token
                    output_tokens += 1

        total_time = time.perf_counter() - t_start
        tpot = (total_time - ttft) / max(output_tokens - 1, 1) if output_tokens > 1 else None

        return RequestResult(
            profile=request["profile"],
            target_isl=request["target_isl"],
            target_osl=request["target_osl"],
            ttft=ttft,
            total_time=total_time,
            output_tokens=output_tokens,
            tpot=tpot,
            success=True,
        )

    except Exception as e:
        return RequestResult(
            profile=request["profile"],
            target_isl=request["target_isl"],
            target_osl=request["target_osl"],
            ttft=None, total_time=None,
            output_tokens=None, tpot=None,
            success=False, error=str(e),
        )


async def run_benchmark(
    requests: list,
    concurrency: int,
    server_url: str = f"http://localhost:{SERVER_PORT}",
) -> List[RequestResult]:
    """以指定并发度发送所有请求"""
    semaphore = asyncio.Semaphore(concurrency)
    results   = []

    async def bounded_request(req):
        async with semaphore:
            return await send_request(session, req, server_url)

    connector = aiohttp.TCPConnector(limit=concurrency + 10)
    async with aiohttp.ClientSession(connector=connector) as session:
        tasks = [bounded_request(r) for r in requests]
        total = len(tasks)
        for i, coro in enumerate(asyncio.as_completed(tasks)):
            result = await coro
            results.append(result)
            if (i + 1) % 10 == 0:
                done = sum(1 for r in results if r.success)
                print(f"\r  进度: {i+1}/{total} ({done} 成功)", end='')
    print()
    return results


def compute_metrics(results: List[RequestResult], slo_ttft: float = 2.0) -> dict:
    """计算汇总指标"""
    ok = [r for r in results if r.success and r.ttft is not None]
    if not ok:
        return {"error": "no successful results"}

    ttfts  = [r.ttft for r in ok]
    tpots  = [r.tpot for r in ok if r.tpot]
    totals = [r.total_time for r in ok]

    return {
        "n_requests":    len(results),
        "n_success":     len(ok),
        "success_rate":  len(ok) / len(results),
        # TTFT
        "ttft_p50":  np.percentile(ttfts, 50),
        "ttft_p95":  np.percentile(ttfts, 95),
        "ttft_p99":  np.percentile(ttfts, 99),
        "ttft_mean": np.mean(ttfts),
        # TPOT
        "tpot_p50":  np.percentile(tpots, 50) if tpots else None,
        "tpot_p95":  np.percentile(tpots, 95) if tpots else None,
        # Throughput
        "total_time":  max(totals),
        "throughput_req_s": len(ok) / max(totals),
        "throughput_tok_s": sum(
            r.output_tokens for r in ok if r.output_tokens
        ) / max(totals),
        # SLO
        "slo_threshold": slo_ttft,
        "slo_rate": sum(1 for t in ttfts if t < slo_ttft) / len(ttfts),
    }


print("Benchmark 函数定义完成")

In [ ]:
import json
import numpy
print(numpy.__version__)

import pandas as pd
from datetime import datetime
CONCURRENCY_LEVELS = [1, 4, 8, 16]    # 模拟不同QPS压力
N_REQUESTS_PER_RUN = 50           # 每组请求数
PROFILES_TO_TEST   = ["inline_completion", "code_explanation",
                       "function_generation"]

import asyncio
import nest_asyncio
nest_asyncio.apply()
async def run_all_experiments():
    all_results = []
    total_runs = len(PROFILES_TO_TEST) * len(CONCURRENCY_LEVELS)
    run_count = 0

    print(f"开始实验：{total_runs} 组 × {N_REQUESTS_PER_RUN} 请求")
    print(f"预计时间：20-30 分钟\n")
    print("=" * 60)

    for profile in PROFILES_TO_TEST:
        for concurrency in CONCURRENCY_LEVELS:
            run_count += 1
            slo = WORKLOAD_PROFILES[profile]["TTFT_SLO"]

            print(f"\n[{run_count}/{total_runs}] Profile={profile}, "
                  f"Concurrency={concurrency}")

            requests = sample_workload(
                profile, N_REQUESTS_PER_RUN, seed=run_count
            )

            t0 = time.time()
            results = await run_benchmark(requests, concurrency)
            elapsed = time.time() - t0

            metrics = compute_metrics(results, slo_ttft=slo)
            metrics.update({
                "experiment_id": run_count,
                "profile": profile,
                "concurrency": concurrency,
                "serving_mode": "standard",
                "model": MODEL_NAME,
                "quantization": QUANTIZATION or "none",
                "wall_time": elapsed,
                "timestamp": datetime.now().isoformat(),
            })
            all_results.append(metrics)

            print(f"  TTFT P50={metrics['ttft_p50']*1000:.0f}ms  "
                  f"P99={metrics['ttft_p99']*1000:.0f}ms  "
                  f"SLO达标={metrics['slo_rate']*100:.0f}%  "
                  f"吞吐={metrics['throughput_req_s']:.1f}req/s")

    # 混合 workload
    print(f"\n[mixed] 混合Workload, Concurrency=4")
    mixed_reqs = sample_mixed_workload(N_REQUESTS_PER_RUN, seed=999)
    mixed_res  = await run_benchmark(mixed_reqs, concurrency=4)
    mixed_m    = compute_metrics(mixed_res, slo_ttft=1.5)
    mixed_m.update({
        "experiment_id": 99, "profile": "mixed",
        "concurrency": 4, "serving_mode": "standard",
        "model": MODEL_NAME, "quantization": QUANTIZATION or "none",
        "timestamp": datetime.now().isoformat(),
    })
    all_results.append(mixed_m)
    print(f"  TTFT P50={mixed_m['ttft_p50']*1000:.0f}ms  "
          f"SLO达标={mixed_m['slo_rate']*100:.0f}%")

    print(f"\n实验完成，共 {len(all_results)} 组结果")
    return all_results


# 运行
all_results = asyncio.get_event_loop().run_until_complete(run_all_experiments())

In [ ]:
import pandas as pd
import json
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M")
csv_path  = f"{DRIVE_RESULTS_DIR}/standard_baseline_{timestamp}.csv"
json_path = f"{DRIVE_RESULTS_DIR}/standard_baseline_{timestamp}.json"

df = pd.DataFrame(all_results)
df.to_csv(csv_path,  index=False)

with open(json_path, 'w') as f:
    json.dump(all_results, f, indent=2, default=str)

print(f"✅ 结果已保存:")
print(f"   CSV:  {csv_path}")
print(f"   JSON: {json_path}")

# 预览结果表
display_cols = [
    'profile', 'concurrency',
    'ttft_p50', 'ttft_p99', 'slo_rate', 'throughput_req_s'
]
preview = df[display_cols].copy()
preview['ttft_p50'] = (preview['ttft_p50'] * 1000).round(0).astype(int).astype(str) + 'ms'
preview['ttft_p99'] = (preview['ttft_p99'] * 1000).round(0).astype(int).astype(str) + 'ms'
preview['slo_rate'] = (preview['slo_rate'] * 100).round(1).astype(str) + '%'
preview['throughput_req_s'] = preview['throughput_req_s'].round(2)

print("\n结果预览：")
print(preview.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

plt.rcParams.update({'font.size': 11, 'figure.dpi': 120})
COLORS = ['#2196F3', '#FF9800', '#4CAF50']
PROFILES = ["inline_completion", "code_explanation", "function_generation"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(
    f"Standard Serving Baseline — {MODEL_NAME}\n"
    "Code Generation Workload Characterization",
    fontsize=13, fontweight='bold'
)

# ── 图1：TTFT P50/P99 by Profile & Concurrency ───────────
ax = axes[0]
x    = np.arange(len(CONCURRENCY_LEVELS))
w    = 0.25
for i, profile in enumerate(PROFILES):
    pdata = df[df['profile'] == profile].sort_values('concurrency')
    p50s  = pdata['ttft_p50'].values * 1000   # ms
    p99s  = pdata['ttft_p99'].values * 1000
    bars  = ax.bar(x + i*w, p50s, w, color=COLORS[i],
                   alpha=0.8, label=profile.replace('_', ' '))
    ax.errorbar(x + i*w, p50s, yerr=[np.zeros(len(p50s)), p99s - p50s],
                fmt='none', color='black', capsize=3, lw=1.5)

ax.set_title('TTFT: P50 (bar) + P99 (error bar)')
ax.set_xlabel('Concurrency')
ax.set_ylabel('TTFT (ms)')
ax.set_xticks(x + w)
ax.set_xticklabels(CONCURRENCY_LEVELS)
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.3)

# ── 图2：SLO 达标率 vs Concurrency ────────────────────────
ax = axes[1]
for i, profile in enumerate(PROFILES):
    pdata = df[df['profile'] == profile].sort_values('concurrency')
    ax.plot(
        pdata['concurrency'], pdata['slo_rate'] * 100,
        marker='o', color=COLORS[i], lw=2,
        label=f"{profile.replace('_',' ')} (SLO={WORKLOAD_PROFILES[profile]['TTFT_SLO']}s)"
    )

ax.axhline(90, color='red', ls='--', lw=1.5, label='90% SLO target')
ax.set_title('SLO Satisfaction Rate vs Concurrency')
ax.set_xlabel('Concurrency')
ax.set_ylabel('SLO Satisfaction Rate (%)')
ax.set_ylim(0, 105)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# ── 图3：Throughput vs Concurrency ────────────────────────
ax = axes[2]
for i, profile in enumerate(PROFILES):
    pdata = df[df['profile'] == profile].sort_values('concurrency')
    ax.plot(
        pdata['concurrency'], pdata['throughput_req_s'],
        marker='s', color=COLORS[i], lw=2,
        label=profile.replace('_', ' ')
    )

ax.set_title('Throughput vs Concurrency')
ax.set_xlabel('Concurrency')
ax.set_ylabel('Throughput (req/s)')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

plt.tight_layout()

# 保存图到 Drive
fig_path = f"{DRIVE_RESULTS_DIR}/baseline_plot_{timestamp}.png"
plt.savefig(fig_path, bbox_inches='tight', dpi=150)
print(f"图表已保存: {fig_path}")
plt.show()

In [ ]:
import numpy as np
import asyncio
import nest_asyncio
nest_asyncio.apply()
ISL_VALUES  = [128, 256, 512, 768, 1024, 1536, 2048, 4096]
FIXED_OSL   = 250     # 固定输出长度
FIXED_CONC  = 8       # 固定并发
N_PER_ISL   = 20      # 每个ISL点的请求数

isl_sweep_results = []

print(f"ISL Sweep 实验: {ISL_VALUES}")
print(f"固定 OSL={FIXED_OSL}, 并发={FIXED_CONC}\n")

for isl in ISL_VALUES:
    requests = [{
        "prompt":     generate_prompt(isl),
        "max_tokens": FIXED_OSL,
        "target_isl": isl,
        "target_osl": FIXED_OSL,
        "profile":    "isl_sweep",
    } for _ in range(N_PER_ISL)]

    results = asyncio.get_event_loop().run_until_complete(
    run_benchmark(requests, FIXED_CONC)
)
    metrics = compute_metrics(results, slo_ttft=1.0)
    metrics["isl"] = isl
    isl_sweep_results.append(metrics)

    print(f"  ISL={isl:4d}: TTFT_P50={metrics['ttft_p50']*1000:.0f}ms  "
          f"P99={metrics['ttft_p99']*1000:.0f}ms")

# 保存 + 可视化
isl_df = pd.DataFrame(isl_sweep_results)
isl_df.to_csv(f"{DRIVE_RESULTS_DIR}/isl_sweep_{timestamp}.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(isl_df['isl'], isl_df['ttft_p50']*1000,
        marker='o', color='#2196F3', lw=2, label='P50')
ax.plot(isl_df['isl'], isl_df['ttft_p99']*1000,
        marker='s', color='#F44336', lw=2, ls='--', label='P99')
ax.axhline(1000, color='orange', ls=':', lw=1.5, label='1s SLO')
ax.fill_between(isl_df['isl'], isl_df['ttft_p50']*1000,
                isl_df['ttft_p99']*1000, alpha=0.1, color='#2196F3')
ax.set_title('TTFT vs Input Sequence Length (ISL)\nStandard Serving, OSL=150, Concurrency=4')
ax.set_xlabel('Input Sequence Length (tokens)')
ax.set_ylabel('TTFT (ms)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{DRIVE_RESULTS_DIR}/isl_sweep_{timestamp}.png", dpi=150)
plt.show()
print("ISL Sweep 完成")

In [ ]:
server_proc.terminate()
server_proc.wait()
print("vLLM 服务已关闭")
print(f"\n所有结果保存在: {DRIVE_RESULTS_DIR}")
!ls -lh {DRIVE_RESULTS_DIR}